<a href="https://colab.research.google.com/github/sravananambiar20/Sentiment_Analysis_of_Product_Reviews/blob/main/code/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-fine-food-reviews' dataset.
Path to dataset files: /kaggle/input/amazon-fine-food-reviews


In [26]:
import pandas as pd
import os

# Assuming the 'path' variable from the previous cell is available and contains the dataset directory.
# A common file in the 'amazon-fine-food-reviews' dataset is 'Reviews.csv'.
df = pd.read_csv(os.path.join(path, 'Reviews.csv'))

df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,26
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,27
Text,0


In [27]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

# Convert Ratings to Sentiment Labels

In [28]:
def sentiment_label(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['Sentiment'] = df['Score'].apply(sentiment_label)

df[['Score', 'Sentiment']].head()

,Score,Sentiment
0,5,positive
1,1,negative
2,4,positive
3,2,negative
4,5,positive


In [29]:
df = df[['Text', 'Sentiment']]

df.head()

,Text,Sentiment
0,I have bought several of the Vitality canned d...,positive
1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,This is a confection that has been around a fe...,positive
3,If you are looking for the secret ingredient i...,negative
4,Great taffy at a great price. There was a wid...,positive


# Check Class Distribution

In [30]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,443777
negative,82037
neutral,42640


# Reduce Dataset Size

In [31]:
df = df.sample(20000, random_state=42)

In [32]:
df = df.reset_index(drop=True)

# Basic Text Cleaning

In [33]:
!pip install nltk

In [34]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [36]:
df['Cleaned_Text'] = df['Text'].apply(clean_text)

df.head()

,Text,Sentiment,Cleaned_Text
0,Having tried a couple of other brands of glute...,positive,tried couple brands glutenfree sandwich cookie...
1,My cat loves these treats. If ever I can't fin...,positive,cat loves treats ever cant find house pop top ...
2,A little less than I expected. It tends to ha...,neutral,little less expected tends muddy taste expecte...
3,"First there was Frosted Mini-Wheats, in origin...",negative,first frosted miniwheats original size frosted...
4,and I want to congratulate the graphic artist ...,positive,want congratulate graphic artist putting entir...


# Encode Labels

In [37]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Sentiment'])

df.head()

,Text,Sentiment,Cleaned_Text,label
0,Having tried a couple of other brands of glute...,positive,tried couple brands glutenfree sandwich cookie...,2
1,My cat loves these treats. If ever I can't fin...,positive,cat loves treats ever cant find house pop top ...,2
2,A little less than I expected. It tends to ha...,neutral,little less expected tends muddy taste expecte...,1
3,"First there was Frosted Mini-Wheats, in origin...",negative,first frosted miniwheats original size frosted...,0
4,and I want to congratulate the graphic artist ...,positive,want congratulate graphic artist putting entir...,2


# Train Test Split

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [39]:
train_data = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

test_data = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

In [40]:
from google.colab import files

files.download("train_data.csv")
files.download("test_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

MODEL TRAINING


In [41]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging
logging.set_verbosity_error()

# ── 1. BERT ──────────────────────────────────────────────────────────────────
from transformers import BertTokenizer, BertForSequenceClassification
import torch

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3,
    ignore_mismatched_sizes=True
)

def bert_tokenize(texts):
    return bert_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

def bert_train(dataloader, epochs=3):
    bert_model.train()
    for epoch in range(epochs):
        for batch in dataloader:
            bert_optimizer.zero_grad()
            outputs = bert_model(input_ids=batch['input_ids'],
                                 attention_mask=batch['attention_mask'],
                                 labels=batch['labels'])
            outputs.loss.backward()
            bert_optimizer.step()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [42]:
# ── 2. DistilBERT ─────────────────────────────────────────────────────────────
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    ignore_mismatched_sizes=True
)

def distilbert_tokenize(texts):
    return distilbert_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

distilbert_optimizer = torch.optim.AdamW(distilbert_model.parameters(), lr=2e-5)

def distilbert_train(dataloader, epochs=3):
    distilbert_model.train()
    for epoch in range(epochs):
        for batch in dataloader:
            distilbert_optimizer.zero_grad()
            outputs = distilbert_model(input_ids=batch['input_ids'],
                                       attention_mask=batch['attention_mask'],
                                       labels=batch['labels'])
            outputs.loss.backward()
            distilbert_optimizer.step()




Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [43]:
# ── 3. SVM with TF-IDF ───────────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('svm', SVC(kernel='linear', C=1.0, probability=True))
])

def svm_train(X_train, y_train):
    svm_model.fit(X_train, y_train)

def svm_predict(X_test):
    return svm_model.predict(X_test)



In [44]:
# ── 4. Logistic Regression with TF-IDF ───────────────────────────────────────
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('lr', LogisticRegression(max_iter=1000, C=1.0, multi_class='multinomial', solver='lbfgs'))
])

def lr_train(X_train, y_train):
    lr_model.fit(X_train, y_train)

def lr_predict(X_test):
    return lr_model.predict(X_test)

In [45]:
import pickle

models = {
    'bert_model': bert_model,
    'distilbert_model': distilbert_model,
    'svm_model': svm_model,
    'lr_model': lr_model
}

with open('models.pkl', 'wb') as f:
    pickle.dump(models, f)

print("All models saved to models.pkl")

All models saved to models.pkl


In [22]:
# Train SVM and Logistic Regression
print("Training SVM...")
svm_train(X_train, y_train)
print("SVM done!")

print("Training Logistic Regression...")
lr_train(X_train, y_train)
print("LR done!")

Training SVM...
SVM done!
Training Logistic Regression...
LR done!


In [23]:
from sklearn.metrics import classification_report, accuracy_score

# SVM Evaluation
svm_preds = svm_predict(X_test)
print("=== SVM Results ===")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print(classification_report(y_test, svm_preds, target_names=['negative', 'neutral', 'positive']))

# LR Evaluation
lr_preds = lr_predict(X_test)
print("=== Logistic Regression Results ===")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds, target_names=['negative', 'neutral', 'positive']))

=== SVM Results ===
Accuracy: 0.83975
              precision    recall  f1-score   support

    negative       0.70      0.56      0.62       581
     neutral       0.63      0.05      0.10       317
    positive       0.86      0.97      0.91      3102

    accuracy                           0.84      4000
   macro avg       0.73      0.53      0.55      4000
weighted avg       0.82      0.84      0.81      4000

=== Logistic Regression Results ===
Accuracy: 0.8415
              precision    recall  f1-score   support

    negative       0.76      0.50      0.61       581
     neutral       0.60      0.07      0.12       317
    positive       0.85      0.98      0.91      3102

    accuracy                           0.84      4000
   macro avg       0.74      0.52      0.55      4000
weighted avg       0.82      0.84      0.81      4000



In [24]:
from torch.utils.data import Dataset, DataLoader

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(list(texts), padding=True, truncation=True, max_length=128)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Use a small sample to save time
X_train_sample = X_train[:2000]
y_train_sample = y_train[:2000]

print("Training DistilBERT...")
distilbert_dataset = ReviewDataset(X_train_sample, y_train_sample, distilbert_tokenizer)
distilbert_loader = DataLoader(distilbert_dataset, batch_size=16, shuffle=True)
distilbert_train(distilbert_loader, epochs=1)
print("DistilBERT done!")

print("Training BERT...")
bert_dataset = ReviewDataset(X_train_sample, y_train_sample, bert_tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=16, shuffle=True)
bert_train(bert_loader, epochs=1)
print("BERT done!")

Training DistilBERT...
DistilBERT done!
Training BERT...
BERT done!


In [46]:
# ── Train SVM and LR ──────────────────────────────────────────────
print("Training SVM...")
svm_train(X_train, y_train)
print("SVM done!")

print("Training Logistic Regression...")
lr_train(X_train, y_train)
print("LR done!")

# ── Evaluate SVM and LR ───────────────────────────────────────────
from sklearn.metrics import classification_report, accuracy_score

svm_preds = svm_predict(X_test)
lr_preds = lr_predict(X_test)

print("=== SVM Results ===")
print(classification_report(y_test, svm_preds, target_names=['negative','neutral','positive']))

print("=== LR Results ===")
print(classification_report(y_test, lr_preds, target_names=['negative','neutral','positive']))

# ── Save reports ──────────────────────────────────────────────────
with open("svm_report.txt", "w") as f:
    f.write(classification_report(y_test, svm_preds, target_names=['negative','neutral','positive']))

with open("lr_report.txt", "w") as f:
    f.write(classification_report(y_test, lr_preds, target_names=['negative','neutral','positive']))

# ── Save small_models.pkl ─────────────────────────────────────────
import pickle
small_models = {
    "models": {
        "TF-IDF + SVM": svm_model,
        "TF-IDF + Logistic Regression": lr_model,
    }
}
with open("small_models.pkl", "wb") as f:
    pickle.dump(small_models, f)

print("All saved!")

Training SVM...
SVM done!
Training Logistic Regression...
LR done!
=== SVM Results ===
              precision    recall  f1-score   support

    negative       0.70      0.56      0.62       581
     neutral       0.63      0.05      0.10       317
    positive       0.86      0.97      0.91      3102

    accuracy                           0.84      4000
   macro avg       0.73      0.53      0.55      4000
weighted avg       0.82      0.84      0.81      4000

=== LR Results ===
              precision    recall  f1-score   support

    negative       0.76      0.50      0.61       581
     neutral       0.60      0.07      0.12       317
    positive       0.85      0.98      0.91      3102

    accuracy                           0.84      4000
   macro avg       0.74      0.52      0.55      4000
weighted avg       0.82      0.84      0.81      4000

All saved!


In [47]:
from google.colab import files
files.download("small_models.pkl")
files.download("svm_report.txt")
files.download("lr_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>